In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
print("Starting CPLID dataset preparation and YOLOv11 training")

!pip install ultralytics -q

import os
import shutil
import random
import glob
import yaml
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import xml.etree.ElementTree as ET
from PIL import Image
from ultralytics import YOLO
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings("ignore")


source_root = Path("/kaggle/input/cplid-dataset/InsulatorDataSet-master")
dest_root   = Path("/kaggle/working/CPLID_standard")

(dest_root / "images").mkdir(parents=True, exist_ok=True)
(dest_root / "labels").mkdir(parents=True, exist_ok=True)

for split in ["train", "val", "test"]:
    (dest_root / "images" / split).mkdir(exist_ok=True)
    (dest_root / "labels" / split).mkdir(exist_ok=True)

# Collect all images
normal_dir   = source_root / "Normal_Insulators"   / "images"
defective_dir = source_root / "Defective_Insulators" / "images"

all_images = list(normal_dir.glob("*.jpg")) + list(defective_dir.glob("*.jpg"))
print(f"Total images found: {len(all_images)}")

random.seed(42)
random.shuffle(all_images)

n_train = int(len(all_images) * 0.75)
n_val   = int(len(all_images) * 0.15)

train_imgs = all_images[:n_train]
val_imgs   = all_images[n_train:n_train + n_val]
test_imgs  = all_images[n_train + n_val:]

def find_xml_files(img_stem):
    paths = []
    # Normal
    paths.extend((source_root / "Normal_Insulators" / "labels").glob(f"{img_stem}.xml"))
    # Defective direct
    paths.extend((source_root / "Defective_Insulators" / "labels").glob(f"{img_stem}.xml"))
    # Defective subfolders
    paths.extend((source_root / "Defective_Insulators" / "labels" / "insulator").glob(f"{img_stem}.xml"))
    paths.extend((source_root / "Defective_Insulators" / "labels" / "defect").glob(f"{img_stem}.xml"))
    return [p for p in paths if p.exists()]

def xml_to_yolo(xml_paths, img_w, img_h):
    lines = []
    seen = set()
    for xml_p in xml_paths:
        try:
            tree = ET.parse(xml_p)
            for obj in tree.findall(".//object"):
                name = obj.find("name").text.strip().lower()
                cls = -1
                if "insulator" in name or "normal" in name or "good" in name:
                    cls = 0
                elif any(x in name for x in ["defect","broken","damage","crack","flashover","defective"]):
                    cls = 1
                if cls == -1:
                    continue

                b = obj.find("bndbox")
                xmin,ymin,xmax,ymax = map(float, [b.find(k).text for k in ["xmin","ymin","xmax","ymax"]])

                xc = (xmin + xmax) / 2 / img_w
                yc = (ymin + ymax) / 2 / img_h
                w  = (xmax - xmin) / img_w
                h  = (ymax - ymin) / img_h

                line = f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"
                if line not in seen:
                    seen.add(line)
                    lines.append(line)
        except:
            pass
    return lines

def process(img_path, split):
    stem = img_path.stem
    xmls = find_xml_files(stem)
    if not xmls:
        return

    try:
        with Image.open(img_path) as im:
            w, h = im.size
    except:
        return

    yolo = xml_to_yolo(xmls, w, h)

    shutil.copy(img_path, dest_root / "images" / split / img_path.name)
    txt_path = dest_root / "labels" / split / f"{stem}.txt"
    with open(txt_path, "w") as f:
        if yolo:
            f.write("\n".join(yolo) + "\n")

for images, name in [(train_imgs,"train"), (val_imgs,"val"), (test_imgs,"test")]:
    print(f"Converting {name} ({len(images)} images)")
    for p in images:
        process(p, name)

print("\nConversion finished.")
for s in ["train","val","test"]:
    ic = len(list((dest_root/"images"/s).glob("*.jpg")))
    lc = len(list((dest_root/"labels"/s).glob("*.txt")))
    print(f"{s:5} → images: {ic:4}   labels: {lc:4}")

data_yaml_path = "/kaggle/working/data.yaml"

data_config = {
    "path": str(dest_root),
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "nc": 2,
    "names": ["insulator", "defect"]
}

with open(data_yaml_path, "w") as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("\ndata.yaml created:")
!cat {data_yaml_path}


model = YOLO("yolo11n.pt")
model.info()

model.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    plots=True,
    val=True,
    augment=True,          
)

val_results = model.val(
    data=data_yaml_path,
    split="val",
    iou=0.5,
    conf=0.25,
    plots=True
)

print("\nValidation Metrics")
print(f"mAP@0.5 : {val_results.box.map50:.4f}")
print(f"Precision : {val_results.box.mp:.4f}")
print(f"Recall : {val_results.box.mr:.4f}")

# ROC Curve
val_dir = sorted(glob.glob("runs/detect/val*"))[-1]
stats = val_results.stats
conf = np.array(stats.get("conf", []))
pred_cls = np.array(stats.get("pred_cls", []))

if len(conf) > 0 and len(pred_cls) > 0:
    y_true = pred_cls.astype(int)
    y_score = conf
    y_true_bin = label_binarize(y_true, classes=[0,1])
    plt.figure(figsize=(7,6))
    for i, name in enumerate(["insulator", "defect"]):
        if y_true_bin[:, i].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score)
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.2f})")
    plt.plot([0,1], [0,1], "--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve (Validation)")
    plt.legend()
    plt.grid()
    plt.show()
else:
    print("ROC skipped: no valid detections")

# Test evaluation
test_results = model.val(
    data=data_yaml_path,
    split="test",
    iou=0.5,
    conf=0.25,
    plots=True
)

print("\nTest Metrics")
print(f"mAP@0.5 : {test_results.box.map50:.4f}")
print(f"Precision : {test_results.box.mp:.4f}")
print(f"Recall : {test_results.box.mr:.4f}")

test_dir = sorted(glob.glob("runs/detect/val*"))[-1]

for fname, title in [("PR_curve.png", "Precision–Recall Curve (Test)"),
                     ("confusion_matrix.png", "Confusion Matrix (Test)")]:
    p = os.path.join(test_dir, fname)
    if os.path.exists(p):
        plt.imshow(plt.imread(p))
        plt.title(title)
        plt.axis("off")
        plt.show()

print("\nTraining & evaluation completed.")